# Parallel Qoruyo OCR for the Jacob of Serugh volumes\n\nThis notebook partitions one volume into deterministic, non-overlapping parts so that several Kaggle sessions can OCR it in parallel.\n\n1. In Kaggle, attach a dataset containing the cropped page images and the two Qoruyo model files named below.\n2. Set `VOLUME`, `PART_INDEX`, and `N_PARTS` in the configuration section. Use the same `N_PARTS` in every session and a different `PART_INDEX` in each.\n3. Run both code cells. Download the ZIP created in `/kaggle/working/` when processing finishes.\n4. Combine the ZIP files from all parts of the volume. Output filenames carry volume-wide page numbers and do not overlap.\n\nKaggle mounts attached datasets below `/kaggle/input`; that platform path is intentionally retained. The notebook discovers the attached dataset rather than embedding an account name or private dataset path. Volume 6 supports two sequential source folders because its scans were split across two image collections.

In [ ]:
%pip install kraken\n

In [ ]:
# Run this once in a separate Kaggle cell:
# !pip install kraken

import math
import os
import re
import subprocess
import zipfile
from collections import Counter
from pathlib import Path


# =========================
# EDIT THIS FOR EACH RUN
# =========================
VOLUME = 1       # 1, 2, 3, 4, 5, or 6
PART_INDEX = 1   # 1 through N_PARTS
N_PARTS = 4


# =========================
# Kaggle paths
# =========================
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

SEG_MODEL_NAME = "syr_print_seg03_91.mlmodel"
OCR_MODEL_NAME = "SyrEastSyr_01_18.mlmodel"

OUT_DIR = (
    WORKING_ROOT
    / f"jacob_vol{VOLUME}_ocr_output_part_{PART_INDEX}"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert KAGGLE_INPUT.exists(), (
    f"Kaggle input directory does not exist: {KAGGLE_INPUT}"
)


# =========================
# Helpers
# =========================
def normalize_component(value):
    """
    Normalize directory names.

    Examples:
        Vol6 photos_1 -> vol6photos1
        Volume-6-Photos-2 -> volume6photos2
    """
    return re.sub(r"[^a-z0-9]", "", value.lower())


def find_required_file(root, filename):
    matches = sorted(root.rglob(filename))

    assert matches, (
        f"Could not find {filename} anywhere under {root}"
    )

    if len(matches) > 1:
        print(f"Multiple matches found for {filename}:")
        for match in matches:
            print(f"  {match}")
        print(f"Using: {matches[0]}")

    return matches[0]


def get_volume_source_part(path, volume):
    """
    Return the source-folder number for an image.

    Volumes 1-5 use one source folder.
    Volume 6 uses two sequential source folders.
    """
    normalized_parts = {
        normalize_component(part)
        for part in path.parts
    }

    if volume == 6:
        part_1_names = {
            "vol6photos1",
            "volume6photos1",
        }

        part_2_names = {
            "vol6photos2",
            "volume6photos2",
        }

        if normalized_parts.intersection(part_1_names):
            return 1

        if normalized_parts.intersection(part_2_names):
            return 2

        return None

    allowed_names = {
        f"vol{volume}photos",
        f"volume{volume}photos",
    }

    if normalized_parts.intersection(allowed_names):
        return 1

    return None


def extract_page_number(path):
    """
    Use the final number in the filename as its local page number.

    Examples:
        page_123.jpg -> 123
        IMG_0042.JPG -> 42
        scan-87.png  -> 87
    """
    numbers = re.findall(r"\d+", path.stem)

    if not numbers:
        return None

    return int(numbers[-1])


def natural_sort_key(path):
    """
    Sort names naturally, so page_2 precedes page_10.
    """
    pieces = re.split(r"(\d+)", path.name.lower())

    return tuple(
        (0, int(piece))
        if piece.isdigit()
        else (1, piece)
        for piece in pieces
    )


def chunks(items, size):
    """
    Yield successive batches of at most `size` items.
    """
    if size <= 0:
        raise ValueError("Batch size must be greater than zero")

    for index in range(0, len(items), size):
        yield items[index:index + size]


# =========================
# Locate dataset and models
# =========================
dataset_candidates = []

for possible_root in KAGGLE_INPUT.iterdir():
    if not possible_root.is_dir():
        continue

    if any(possible_root.rglob(SEG_MODEL_NAME)):
        dataset_candidates.append(possible_root)

assert dataset_candidates, (
    f"Could not find {SEG_MODEL_NAME} under {KAGGLE_INPUT}.\n"
    "Available Kaggle dataset directories:\n"
    + "\n".join(str(path) for path in KAGGLE_INPUT.iterdir())
)

if len(dataset_candidates) > 1:
    print("Multiple possible dataset roots found:")
    for candidate in dataset_candidates:
        print(f"  {candidate}")
    print(f"Using: {dataset_candidates[0]}")

DATASET_ROOT = dataset_candidates[0]

SEG_MODEL = find_required_file(
    DATASET_ROOT,
    SEG_MODEL_NAME,
)

OCR_MODEL = find_required_file(
    DATASET_ROOT,
    OCR_MODEL_NAME,
)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Segmentation model: {SEG_MODEL}")
print(f"OCR model: {OCR_MODEL}")


# =========================
# Discover volume images
# =========================
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".webp",
    ".bmp",
}

discovered_images = []

for path in DATASET_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    source_part = get_volume_source_part(path, VOLUME)

    if source_part is None:
        continue

    discovered_images.append((source_part, path))

assert discovered_images, (
    f"No images found for Volume {VOLUME} under {DATASET_ROOT}"
)


# Sort by source folder first, then by local page number.
discovered_images.sort(
    key=lambda item: (
        item[0],
        (
            extract_page_number(item[1])
            if extract_page_number(item[1]) is not None
            else float("inf")
        ),
        natural_sort_key(item[1]),
    )
)


# =========================
# Assign local page numbers
# =========================
source_part_counters = Counter()
image_records = []

for source_part, path in discovered_images:
    source_part_counters[source_part] += 1

    local_page_number = extract_page_number(path)

    if local_page_number is None:
        local_page_number = source_part_counters[source_part]

    image_records.append({
        "source_part": source_part,
        "local_page_number": local_page_number,
        "path": path,
    })


# =========================
# Create unique volume-wide
# page numbers
# =========================
if VOLUME == 6:
    part_1_records = [
        record
        for record in image_records
        if record["source_part"] == 1
    ]

    part_2_records = [
        record
        for record in image_records
        if record["source_part"] == 2
    ]

    assert part_1_records, (
        "Volume 6 folder 'Vol6 photos_1' was not found or was empty"
    )

    assert part_2_records, (
        "Volume 6 folder 'Vol6 photos_2' was not found or was empty"
    )

    # The second source PDF restarts at page 0.
    # Offset it so that it follows the final local page in part 1.
    part_2_offset = (
        max(
            record["local_page_number"]
            for record in part_1_records
        )
        + 1
    )

    print(f"Volume 6 part-2 offset: {part_2_offset}")

    for record in image_records:
        if record["source_part"] == 1:
            record["global_page_number"] = (
                record["local_page_number"]
            )
        else:
            record["global_page_number"] = (
                part_2_offset
                + record["local_page_number"]
            )

else:
    for record in image_records:
        record["global_page_number"] = (
            record["local_page_number"]
        )


# Confirm that every output page number is unique.
global_page_numbers = [
    record["global_page_number"]
    for record in image_records
]

duplicate_global_pages = [
    page_number
    for page_number, count
    in Counter(global_page_numbers).items()
    if count > 1
]

assert not duplicate_global_pages, (
    "Duplicate volume-wide page numbers remain: "
    f"{duplicate_global_pages[:20]}"
)


# Final deterministic volume ordering.
image_records.sort(
    key=lambda record: (
        record["global_page_number"],
        natural_sort_key(record["path"]),
    )
)


# Each tuple contains:
# sequence_number,
# global_page_number,
# local_page_number,
# source_part,
# image_path
pages = []

for sequence_number, record in enumerate(
    image_records,
    start=1,
):
    pages.append((
        sequence_number,
        record["global_page_number"],
        record["local_page_number"],
        record["source_part"],
        record["path"],
    ))


print(f"\nFound total Volume {VOLUME} images: {len(pages)}")
print(f"First image: {pages[0][4]}")
print(f"Last image:  {pages[-1][4]}")

print("\nFirst 10 ordered images:")

for (
    sequence_number,
    global_page,
    local_page,
    source_part,
    path,
) in pages[:10]:
    print(
        f"  sequence={sequence_number}, "
        f"global_page={global_page}, "
        f"source_part={source_part}, "
        f"local_page={local_page}: "
        f"{path.name}"
    )

if VOLUME == 6:
    source_switches = []

    for index in range(1, len(pages)):
        previous_part = pages[index - 1][3]
        current_part = pages[index][3]

        if previous_part != current_part:
            source_switches.append({
                "sequence": pages[index][0],
                "global_page": pages[index][1],
                "path": str(pages[index][4]),
            })

    print("\nVolume 6 source-folder transition:")
    print(source_switches)


# =========================
# Select one parallel part
# =========================
assert 1 <= PART_INDEX <= N_PARTS, (
    f"PART_INDEX must be between 1 and {N_PARTS}"
)

part_size = math.ceil(len(pages) / N_PARTS)

start_index = (PART_INDEX - 1) * part_size
end_index = min(
    PART_INDEX * part_size,
    len(pages),
)

selected_pages = pages[start_index:end_index]

assert selected_pages, (
    f"No pages selected for Volume {VOLUME}, "
    f"part {PART_INDEX}/{N_PARTS}"
)

print(
    f"\nRunning Volume {VOLUME}, "
    f"part {PART_INDEX}/{N_PARTS}"
)
print(f"Selected images: {len(selected_pages)}")
print(f"Global sequence range: {start_index + 1}-{end_index}")
print(f"First selected image: {selected_pages[0][4]}")
print(f"Last selected image:  {selected_pages[-1][4]}")


# =========================
# Kraken OCR runner
# =========================
env = os.environ.copy()
env["TF_CPP_MIN_LOG_LEVEL"] = "3"

BATCH_SIZE = 50
failures = []

for batch_index, batch in enumerate(
    chunks(selected_pages, BATCH_SIZE),
    start=1,
):
    cmd = ["kraken"]
    pages_to_run = []

    for (
        sequence_number,
        global_page_number,
        local_page_number,
        source_part,
        page_path,
    ) in batch:
        output_txt = OUT_DIR / (
            f"Jacob_Bedjan_Vol{VOLUME}"
            f"_page_{global_page_number}.txt"
        )

        # Safe resumption after an interrupted Kaggle run.
        if output_txt.exists() and output_txt.stat().st_size > 0:
            continue

        cmd.extend([
            "-i",
            str(page_path),
            str(output_txt),
        ])

        pages_to_run.append({
            "sequence_number": sequence_number,
            "global_page_number": global_page_number,
            "local_page_number": local_page_number,
            "source_part": source_part,
            "page_path": page_path,
            "output_txt": output_txt,
        })

    if not pages_to_run:
        print(
            f"Batch {batch_index}: "
            "all selected pages already OCRed; skipping"
        )
        continue

    cmd.extend([
        "segment",
        "-bl",
        "-i",
        str(SEG_MODEL),
        "-d",
        "horizontal-rl",

        "ocr",
        "-m",
        str(OCR_MODEL),
        "--reorder",
        "--base-dir",
        "R",
    ])

    first_record = pages_to_run[0]
    last_record = pages_to_run[-1]

    print(
        f"Running batch {batch_index}: "
        f"global pages "
        f"{first_record['global_page_number']}-"
        f"{last_record['global_page_number']}"
    )

    try:
        subprocess.run(
            cmd,
            check=True,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )

        print(
            f"Finished batch {batch_index}: "
            f"{len(pages_to_run)} image(s)"
        )

    except subprocess.CalledProcessError as error:
        print(
            f"FAILED batch {batch_index}: "
            f"global pages "
            f"{first_record['global_page_number']}-"
            f"{last_record['global_page_number']}"
        )

        print("STDOUT:")
        print(error.stdout)

        print("STDERR:")
        print(error.stderr)

        failures.append({
            "batch": batch_index,
            "first_global_page": (
                first_record["global_page_number"]
            ),
            "last_global_page": (
                last_record["global_page_number"]
            ),
            "returncode": error.returncode,
        })


# =========================
# Validate generated files
# =========================
txt_files = sorted(
    OUT_DIR.glob(
        f"Jacob_Bedjan_Vol{VOLUME}_page_*.txt"
    ),
    key=lambda path: (
        extract_page_number(path)
        if extract_page_number(path) is not None
        else float("inf")
    ),
)

expected_output_names = {
    (
        f"Jacob_Bedjan_Vol{VOLUME}"
        f"_page_{global_page_number}.txt"
    )
    for (
        sequence_number,
        global_page_number,
        local_page_number,
        source_part,
        page_path,
    ) in selected_pages
}

actual_output_names = {
    path.name
    for path in txt_files
    if path.stat().st_size > 0
}

missing_output_names = sorted(
    expected_output_names - actual_output_names
)

print(f"\nExpected OCR files: {len(expected_output_names)}")
print(f"Completed OCR files: {len(actual_output_names)}")
print(f"Missing OCR files: {len(missing_output_names)}")
print(f"Failed batches: {len(failures)}")

if missing_output_names:
    print("First missing files:")

    for missing_name in missing_output_names[:20]:
        print(f"  {missing_name}")

if failures:
    print("Batch failures:")
    print(failures)


# =========================
# Package this volume part
# =========================
zip_path = WORKING_ROOT / (
    f"jacob_vol{VOLUME}"
    f"_ocr_output_part_{PART_INDEX}.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for txt_path in txt_files:
        archive.write(
            txt_path,
            arcname=txt_path.name,
        )

print(f"\nCreated ZIP archive: {zip_path}")